Install conda

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


Git clone:

!git clone https://github.com/zhaoyanpeng208/EviDTI.git

%cd EviDTI

Install dependency

In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
-f https://data.pyg.org/whl/torch-2.6.0+cu124.html

!pip install torch-geometric==2.6.1

!pip install rdkit pyyaml yacs pandas scikit-learn tqdm aiohttp requests yarl multidict psutil

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


Feature extraction

1D protein Seqence

In [ ]:
# extract_p_emb.py

In [ ]:
import torch

from transformers import T5EncoderModel, T5Tokenizer
from transformers import BertModel, BertTokenizer
from transformers import XLNetModel, XLNetTokenizer
from transformers import AlbertModel, AlbertTokenizer

import re
import gc
import os
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model_name = "Rostlab/prot_t5_xl_uniref50"

if "t5" in model_name:
    tokenizer = T5Tokenizer.from_pretrained(
        "/content/drive/MyDrive/prot_t5_xl_uniref50",
        do_lower_case=False
    )
    model = T5EncoderModel.from_pretrained(
        "/content/drive/MyDrive/prot_t5_xl_uniref50"
    )
elif "albert" in model_name:
    tokenizer = AlbertTokenizer.from_pretrained(model_name, do_lower_case=False )
    model = AlbertModel.from_pretrained(model_name)
elif "bert" in model_name:
    tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False )
    model = BertModel.from_pretrained(model_name)
elif "xlnet" in model_name:
    tokenizer = XLNetTokenizer.from_pretrained(model_name, do_lower_case=False )
    model = XLNetModel.from_pretrained(model_name)
else:
    print("Unkown model name")


In [ ]:
gc.collect()
print("Number of model parameters is: " + str(int(sum(p.numel() for p in model.parameters())/1000000)) + " Million")
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model = model.eval()
if torch.cuda.is_available():
    model = model.half()


Number of model parameters is: 1208 Million


In [ ]:
def downloadNetsurfpDataset():
    netsurfpDatasetTrainUrl = 'https://www.dropbox.com/s/98hovta9qjmmiby/Train_HHblits.csv?dl=1'
    casp12DatasetValidUrl = 'https://www.dropbox.com/s/te0vn0t7ocdkra7/CASP12_HHblits.csv?dl=1'
    cb513DatasetValidUrl = 'https://www.dropbox.com/s/9mat2fqqkcvdr67/CB513_HHblits.csv?dl=1'
    ts115DatasetValidUrl = 'https://www.dropbox.com/s/68pknljl9la8ax3/TS115_HHblits.csv?dl=1'

    datasetFolderPath = "dataset/"
    trainFilePath = os.path.join(datasetFolderPath, 'Train_HHblits.csv')
    casp12testFilePath = os.path.join(datasetFolderPath, 'CASP12_HHblits.csv')
    cb513testFilePath = os.path.join(datasetFolderPath, 'CB513_HHblits.csv')
    ts115testFilePath = os.path.join(datasetFolderPath, 'TS115_HHblits.csv')
    combinedtestFilePath = os.path.join(datasetFolderPath, 'Validation_HHblits.csv')

    if not os.path.exists(datasetFolderPath):
        os.makedirs(datasetFolderPath)

    def download_file(url, filename):
        response = requests.get(url, stream=True)
        with tqdm.wrapattr(open(filename, "wb"), "write", miniters=1,
                           total=int(response.headers.get('content-length', 0)),
                           desc=filename) as fout:
            for chunk in response.iter_content(chunk_size=4096):
                fout.write(chunk)

    if not os.path.exists(trainFilePath):
        download_file(netsurfpDatasetTrainUrl, trainFilePath)

    if not os.path.exists(casp12testFilePath):
        download_file(casp12DatasetValidUrl, casp12testFilePath)

    if not os.path.exists(cb513testFilePath):
        download_file(cb513DatasetValidUrl, cb513testFilePath)

    if not os.path.exists(ts115testFilePath):
        download_file(ts115DatasetValidUrl, ts115testFilePath)

    if not os.path.exists(combinedtestFilePath):
        combined_csv = pd.concat(
            [pd.read_csv(f) for f in [casp12testFilePath, cb513testFilePath, ts115testFilePath]])
        combined_csv.to_csv(os.path.join(datasetFolderPath, "Validation_HHblits.csv"),
                            index=False,
                            encoding='utf-8-sig')


In [ ]:
def load_dataset(path):
    df = pd.read_csv(path, names=['id', 'input'])
    ids = [id for id in df['id']]
    df['input_fixed'] = ["".join(seq.split()) for seq in df['input']]
    df['input_fixed'] = [re.sub(r"[UZOB]", "X", seq) for seq in df['input_fixed']]
    seqs = [list(seq) for seq in df['input_fixed']]
    return ids, seqs


In [ ]:
DTI_ids, DTI_seqs = load_dataset('/content/drive/MyDrive/target_info.csv')


In [ ]:
def embed_dataset(dataset_seqs, shift_left = 0, shift_right = -1):
    inputs_embedding = []

    for sample in tqdm(dataset_seqs):
        with torch.no_grad():
            ids = tokenizer.batch_encode_plus([sample], add_special_tokens=True, padding=True, is_split_into_words=True, return_tensors="pt")
            embedding = model(input_ids=ids['input_ids'].to(device))[0]
            inputs_embedding.append(embedding[0].detach().cpu().numpy()[shift_left:shift_right])

    return inputs_embedding


In [ ]:
if "t5" in model_name:
    shift_left = 0
    shift_right = -1
elif "bert" in model_name:
    shift_left = 1
    shift_right = -1
elif "xlnet" in model_name:
    shift_left = 0
    shift_right = -2
elif "albert" in model_name:
    shift_left = 1
    shift_right = -1
else:
    print("Unkown model name")


In [ ]:
DTI_seqs_embd = embed_dataset(DTI_seqs, shift_left, shift_right)


  0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
print_idx = 0

print("Original Fasta Sequence : ")
print("".join(DTI_seqs[print_idx]))

print("Generated Sequence Features : ")
print(DTI_seqs_embd[print_idx])


Original Fasta Sequence : 
seq
Generated Sequence Features : 
[[-0.006763 -0.1299   -0.03766  ... -0.04678  -0.03067  -0.00847 ]
 [-0.003944 -0.1302   -0.04025  ... -0.04532  -0.0218   -0.002628]
 [-0.001999 -0.1349   -0.03662  ... -0.04398  -0.02483   0.002064]]


In [ ]:
DTI_seqs_dic = {}
for i in range(len(DTI_seqs)):
    DTI_seqs_dic[DTI_ids[i]] = DTI_seqs_embd[i]

np.save('/content/drive/MyDrive/protein_emb.npy', DTI_seqs_dic)


2D topological graph

In [ ]:
# extract_drug_2d_emb.py

3D spatial structure

In [ ]:
# extract_drug_3d_emb.py